# Suite TSPRS — Parsing TypeScript

The standards of TypeScript and ECMAScript parse source text into TypeScript trees by delegating to
tree-sitter-typescript, then rewriting its concrete syntax tree into the kinds of typescript-estree. The converter
regroups what tree-sitter-typescript 0.23 groups unlike TypeScript, and a pre-pass handles the constructs it cannot
parse. Each case parses constructs and checks the tree (by shape where it matters, and by the text it prints), then
the errors, each with its line and column. In TypeScript, typescript-estree itself checks the conformance sources too.

In [ ]:
from pathlib import Path

from mbse.Programs.Framework import Syntax as F
from mbse.Programs.Framework.Errors import ParseError
from mbse.Programs.TypeScript import ECMAScript2020, ECMAScript2025, Syntax as S, TypeScript50, TypeScript59, _Parser
from mbse.Programs.TypeScript._Standard import TypeScriptStandard
from support import raises

TS = TypeScript59.STANDARD
TSX = TypeScript59.JSX
SOURCES = Path("../../conformance/sources")


def parse(text, standard=TS):
    program = standard.parse(text)
    assert S.LANGUAGE.validate(program) == [], S.LANGUAGE.validate(program)
    return program


def first(text, standard=TS):
    return parse(text, standard).body[0]


def value(text, standard=TS):
    """The expression of the expression statement `text`."""
    return first(text, standard).expression


def same_text(text, expected=None, standard=TS):
    """Parses `text`, checks it prints `expected` (by default itself) and that the print parses to the same tree."""
    program = parse(text, standard)
    printed = standard.print(program)
    assert printed == (text if expected is None else expected) + "\n", printed
    assert F.same(parse(printed, standard), program)
    return program


def fails(text, message, line, column, standard=TS):
    with raises(ParseError, match=message):
        standard.parse(text)
    try:
        standard.parse(text)
    except ParseError as error:
        assert (error.line, error.column) == (line, column), (error.line, error.column)


def kinds(nodes):
    return [type(n).__name__ for n in nodes]

## TSPRS-01 · The conformance sources parse to valid trees, which print and parse back the same, and hold every kind

In [ ]:
seen = set()
for name, standard in (("typescript59.ts", TS), ("typescript59-jsx.tsx", TSX)):
    text = (SOURCES / name).read_text(encoding="utf-8")
    program = parse(text, standard)
    printed = standard.print(program)
    assert F.same(parse(printed, standard), program) and standard.print(parse(printed, standard)) == printed
    parsed, positions, _ = _Parser.parse(text, standard.jsx)
    assert all(id(n) in positions for n in F.walk(parsed))  # every node is placed, for the standard's problems
    seen |= {type(n).__name__ for n in F.walk(program)}
assert seen == set(S.LANGUAGE.kinds())
print(len(seen), "kinds")

## TSPRS-02 · Literals keep their spelling: numbers, bigints, strings, regular expressions and templates

In [ ]:
for literal in ("0", "1_000", "0xFF", "0o17", "0b1010", "1.5e-3", ".5", "5.", "10n", "0xFFn",
                "'a'", '"a"', "'\\n\\u00e9'", "/a+/gi", "/[/]/v", "true", "false", "null"):
    assert F.same(value(literal), S.Literal(raw=literal)), literal
    same_text(f"{literal};")
assert isinstance(value("undefined"), S.Identifier)
template = value("`a${b}c${`d${e}`}`")
assert [q.raw for q in template.quasis] == ["a", "c", ""] and [q.tail for q in template.quasis] == [False, False, True]
assert isinstance(template.expressions[1], S.TemplateLiteral)
assert [q.raw for q in value("`\\n${x}\\``").quasis] == ["\\n", "\\`"]
assert isinstance(value("String.raw`x`"), S.TaggedTemplateExpression)
for text in ("`a${b}c`;", "String.raw`x${y}`;", "`\\n${x}\\``;"):
    same_text(text)
print("ok")

## TSPRS-03 · Expressions, with TypeScript's precedence where tree-sitter-typescript's differs

In [ ]:
grouped = value("a ?? b as T")  # `as` binds tighter than `??`
assert isinstance(grouped, S.LogicalExpression) and isinstance(grouped.right, S.TSAsExpression)
grouped = value("a || b && c as number < d")
assert grouped.operator == "||" and grouped.right.operator == "&&" and grouped.right.right.operator == "<"
assert isinstance(grouped.right.right.left, S.TSAsExpression)
grouped = value("a ? b : c as T")
assert isinstance(grouped, S.ConditionalExpression) and isinstance(grouped.alternate, S.TSAsExpression)
assert value("x = a satisfies T").right.typeAnnotation is not None
assert isinstance(value("f = () => a as T").right.body, S.TSAsExpression)
assert value("a == b as T == c").left.operator == "=="
member = value("x as A.B.C")  # the names after `as` are the type's
assert isinstance(member, S.TSAsExpression) and S.LANGUAGE.validate(member) == []
assert TS.print(member.typeAnnotation) == "A.B.C"
assert TS.print(value("x as A | B.C.D").typeAnnotation) == "A | B.C.D"
assert TS.print(value("x satisfies A & B.C.D").typeAnnotation) == "A & B.C.D"
assert isinstance(value("(x as T).y"), S.MemberExpression)
assert isinstance(value("x as T[]"), S.TSAsExpression)
assert isinstance(value("x as T[].y").object, S.TSAsExpression)  # only names continue a type
constant = value("[1] as const")
assert constant.typeAnnotation.typeName.name == "const"
assert isinstance(value("a ?? b as const").right, S.TSAsExpression)  # `as const` binds as `as` does
assert isinstance(value("a + b as T"), S.TSAsExpression)  # `+` binds tighter than `as`
asserted = value("a ?? b!")
assert isinstance(asserted, S.LogicalExpression) and isinstance(asserted.right, S.TSNonNullExpression)
assert isinstance(value("a ? b : c!").alternate, S.TSNonNullExpression)
assert isinstance(value("x = y!").right, S.TSNonNullExpression)
chain = value("a?.b.c!.d?.[e]?.(f)")
assert isinstance(chain, S.ChainExpression) and chain.expression.optional
assert isinstance(value("a!.b"), S.MemberExpression) and not isinstance(value("a.b"), S.ChainExpression)
assert isinstance(value("(a?.b).c").object, S.ParenthesizedExpression)
assert kinds(value("a, b, c").expressions) == ["Identifier"] * 3
assert value("x++").prefix is False and value("--x").prefix is True
assert value("f<T>(a)").typeArguments is not None and isinstance(value("f<T>"), S.TSInstantiationExpression)
assert isinstance(value("new A<T>"), S.NewExpression) and value("new A").arguments == []
assert isinstance(value("import('m', { with: {} })"), S.ImportExpression) and value("import('m')").options is None
assert isinstance(value("<T>x"), S.TSTypeAssertion)
assert isinstance(value("#x in o").left, S.PrivateIdentifier)
assert value("a **= b").operator == "**=" and value("a >>>= b").operator == ">>>="
assert value("yield* g").delegate is True and value("yield").argument is None
assert value("async (a) => { await a; }").isAsync is True and isinstance(value("a => a").params[0], S.Identifier)
assert kinds(value("[a, , ...b]").elements) == ["Identifier", "Elision", "SpreadElement"]
literal = value("({ a, b: 1, [c]: 2, 'd': 3, 4: 5, m() {}, get g() { return 1; }, set s(v) {}, async *k() {}, ...r })")
properties = literal.expression.properties
assert [p.propertyKind for p in properties[:8]] == ["init", "init", "init", "init", "init", "init", "get", "set"]
assert properties[0].shorthand and properties[2].computed and properties[5].method
assert isinstance(properties[-1], S.SpreadElement)
meta = value("new.target")
assert (meta.meta.name, meta.property.name) == ("new", "target") and value("import.meta").meta.name == "import"
for text in ("a ?? b as T;", "a || b && c as number < d;", "a?.b.c!.d?.[e]?.(f);", "x = (a, b);", "f<T>(a);",
             "new A<T>();", "new.target;", "(x as T).y;", "x as A.B.C;", "<T>x;", "#x in o;", "a **= b;",
             "async (a) => {\n    await a;\n};", "(a) => a;", "a?.(b)!;", "(await a) ** 2;", "(-a) ** 2;"):
    same_text(text)
same_text("new A;", "new A();")
print("ok")

## TSPRS-04 · Statements: declarations of variables, conditions, loops, switches, exceptions, labels and jumps

In [ ]:
declaration = first("let a = 1, [b, , ...c] = d, { e, f: g = 2, ...h } = i;")
assert declaration.declarationKind == "let" and len(declaration.declarations) == 3
pattern = declaration.declarations[2].id
assert kinds(pattern.properties) == ["Property", "Property", "RestElement"]
assert pattern.properties[0].shorthand and isinstance(pattern.properties[1].value, S.AssignmentPattern)
defaulted = first("const { a = 1 } = b;").declarations[0].id.properties[0]
assert defaulted.shorthand and isinstance(defaulted.value, S.AssignmentPattern)
assert first("let x!: number;").declarations[0].definite is True
assert first("var x;").declarations[0].init is None
using = first("using r = f();")
assert using.declarationKind == "using" and first("await using r = f();").declarationKind == "await using"
loop = first("for (using x of y) {}")
assert loop.left.declarationKind == "using" and first("for (await using x of y) {}").left.declarationKind == (
    "await using")
assert first("for await (const x of y) {}").isAwait is True
assert first("for (var x = 1 in o) {}").left.declarations[0].init.raw == "1"
assert isinstance(first("for (x in y) z;").left, S.Identifier)
assert first("for (;;) {}").init is None and first("for (i = 0; i < 1; i++) {}").test is not None
assert isinstance(first("for (let i = 0;;) {}").init, S.VariableDeclaration)
switch = first("switch (a) { case 1: b; break; default: c; }")
assert [c.test is None for c in switch.cases] == [False, True] and len(switch.cases[0].consequent) == 2
assert first("switch (a) { // dropped: before every case\n}").cases == []
attempt = first("try { a; } catch (e: unknown) { b; } finally { c; }")
assert attempt.handler.param.typeAnnotation is not None and attempt.finalizer is not None
assert first("try {} catch {}").handler.param is None and first("try {} finally {}").handler is None
assert first("try {} catch ({ message }) {}").handler.param.properties[0].shorthand
assert first("if (a) b; else if (c) d; else e;").alternate.alternate is not None
assert first("l: for (;;) { break l; continue l; }").body.body.body[0].label.name == "l"
assert first("throw e;").argument.name == "e"
assert isinstance(parse("function f() { return; }").body[0].body.body[0], S.ReturnStatement)
same_text("function f() {\n    return;\n}")
for text in ("let x!: number;", "for (using x of y) {}", "for (await using x of y) {}", "do {} while (a);",
             "do\n    a;\nwhile (b);", "while (a)\n    b;", "with (o) {}", "debugger;", ";", "if (a) {} else {}",
             "switch (a) {\n    case 1:\n        b;\n        break;\n    default:\n        c;\n}",
             "try {} catch (e: unknown) {} finally {}", "l: for (;;) {\n    break l;\n    continue l;\n}",
             "for (x in y)\n    z;", "for (const [a, b] of c) {}"):
    same_text(text)
print("ok")

## TSPRS-05 · Functions and classes: signatures, overloads, members, modifiers, decorators and parameter properties

In [ ]:
function = first("export async function* f<T>(this: X, a?: T, b = 1, ...c: T[]): AsyncGenerator<T> {}").declaration
assert function.isAsync and function.generator and kinds(function.params) == [
    "Identifier", "Identifier", "AssignmentPattern", "RestElement"]
assert function.params[1].optional and function.returnType is not None
assert isinstance(first("declare function f(): void;"), S.TSDeclareFunction)
assert first("declare function f(): void;").declare is True
assert isinstance(first("function f(a: string): string;"), S.TSDeclareFunction)
assert first("function g(x: unknown): x is string { return true; }").returnType.typeAnnotation.parameterName.name == (
    "x")
assert first("function g(x: unknown): asserts x {}").returnType.typeAnnotation.asserts is True
assert first("function g(this: X): asserts this is Y {}").returnType.typeAnnotation.typeAnnotation is not None
klass = first("""@d @e.f()
abstract class C<T> extends B<T> implements I, J<T> {
    #p = 1;
    static q?: number;
    declare readonly r: string;
    protected override s!: number;
    accessor t = 1;
    abstract u: string;
    abstract accessor w: number;
    [k: string]: unknown;
    constructor(private readonly a: T, public b = 1, @g c?: number) { super(); }
    get v(): number { return 1; }
    set v(x) {}
    static { C.q = 1; }
    m?(): void;
    abstract n(): void;
    'constructor'() {}
    [Symbol.iterator]() {}
    @h async *o() {}
}""")
assert klass.abstract and len(klass.decorators) == 2 and len(klass.implements) == 2
assert klass.superTypeArguments is not None
members = klass.body.body
assert kinds(members) == [
    "PropertyDefinition", "PropertyDefinition", "PropertyDefinition", "PropertyDefinition", "AccessorProperty",
    "TSAbstractPropertyDefinition", "TSAbstractAccessorProperty", "TSIndexSignature", "MethodDefinition",
    "MethodDefinition", "MethodDefinition", "StaticBlock", "MethodDefinition", "TSAbstractMethodDefinition",
    "MethodDefinition", "MethodDefinition", "MethodDefinition"]
assert isinstance(members[0].key, S.PrivateIdentifier) and members[1].static and members[1].optional
assert members[2].declare and members[2].readonly and members[3].accessibility == "protected" and members[3].override
assert members[3].definite
constructor = members[8]
assert constructor.methodKind == "constructor" and kinds(constructor.value.params) == [
    "TSParameterProperty", "TSParameterProperty", "Identifier"]
assert constructor.value.params[0].readonly and constructor.value.params[1].accessibility == "public"
assert len(constructor.value.params[2].decorators) == 1
assert [m.methodKind for m in members[9:11]] == ["get", "set"] and members[12].optional
assert isinstance(members[12].value, S.TSEmptyBodyFunctionExpression)
assert members[14].methodKind == "constructor" and members[15].computed
assert len(members[16].decorators) == 1 and members[16].value.isAsync and members[16].value.generator
assert first("class D { static constructor() {} }").body.body[0].methodKind == "method"
heritage = first("class X extends B implements a.b.C, d.E<T> {}")
assert heritage.superTypeArguments is None and TS.print(heritage.implements[0]) == "a.b.C"
assert isinstance(first("interface I extends a.B {}").extends[0].expression, S.MemberExpression)
assert isinstance(value("(class { })"), S.ParenthesizedExpression)
assert isinstance(first("export default class {}").declaration, S.ClassDeclaration)
assert first("export default function () {}").declaration.id is None
assert isinstance(first("export default function* g() {}").declaration, S.FunctionDeclaration)
assert isinstance(first("@d export class E {}").declaration, S.ClassDeclaration)
assert len(first("@d export class E {}").declaration.decorators) == 1
assert len(first("@d export default class {}").declaration.decorators) == 1
for text in ("declare function f(): void;", "function f(a: string): string;",
             "class C {\n    constructor(private readonly a: T, public b = 1) {}\n}",
             "class C {\n    #p = 1;\n    static q?: number;\n    accessor t = 1;\n}", "@d\nexport class E {}",
             "export default class {}", "export default function() {}", "class C {\n    get v(): number {\n"
             "        return 1;\n    }\n    set v(x) {}\n}", "abstract class A {\n    abstract n(): void;\n}",
             "class C {\n    async *o() {}\n}"):
    same_text(text)
print("ok")

## TSPRS-06 · Modules: imports, exports, attributes, import equals, namespaces, ambient modules and `declare global`

In [ ]:
imported = first("import d, { a as b, type c, 'e' as f } from 'm' with { type: 'json' };")
assert kinds(imported.specifiers) == ["ImportDefaultSpecifier", "ImportSpecifier", "ImportSpecifier",
                                      "ImportSpecifier"]
assert imported.specifiers[2].importKind == "type" and isinstance(imported.specifiers[3].imported, S.Literal)
assert imported.attributes[0].key.name == "type"
assert first("import type { T } from 'm';").importKind == "type"
assert first("import * as ns from 'm';").specifiers[0].local.name == "ns"
assert first("import 'side';").specifiers == []
assert first("import x = require('m');").moduleReference.expression.raw == "'m'"
assert first("import type x = require('m');").importKind == "type"
assert isinstance(first("import y = A.B.C;").moduleReference, S.TSQualifiedName)
assert first("export * as ns from 'm';").exported.name == "ns"
assert isinstance(first("import a from 'm' with { 'type': 'json' };").attributes[0].key, S.Literal)
assert isinstance(first("export * as 'n s' from 'm';").exported, S.Literal)
named = first("export { a as b, type c, 'd' } from 'm';")
assert [s.exportKind for s in named.specifiers] == ["value", "type", "value"] and named.source is not None
assert isinstance(named.specifiers[2].exported, S.Literal)
assert first("export { a as 'b c' };").specifiers[0].exported.raw == "'b c'"
assert first("export type { T };").exportKind == "type"
assert first("export interface I {}").exportKind == "type" and first("export declare const x: number;").exportKind == (
    "type")
assert first("export const y = 1;").exportKind == "value"
assert isinstance(first("export = x;"), S.TSExportAssignment)
assert first("export as namespace N;").id.name == "N"
assert isinstance(first("export default a + b;").declaration, S.BinaryExpression)
namespace = first("namespace A.B { export const x = 1; }")
assert isinstance(namespace.id, S.TSQualifiedName) and namespace.moduleKind == "namespace"
assert first("module M {}").moduleKind == "module"
ambient = parse('declare module "m" { export const v: number; }\ndeclare module "short";\nx;').body
assert ambient[0].declare and isinstance(ambient[0].id, S.Literal) and ambient[1].body is None
assert kinds(ambient) == ["TSModuleDeclaration", "TSModuleDeclaration", "ExpressionStatement"]
assert first("declare global { interface W {} }").moduleKind == "global"
in_module = first('declare module "m" { global { interface W {} } }').body.body[0]
assert in_module.moduleKind == "global" and in_module.declare is False
assert first("type A<T> = T[];").typeParameters.params[0].name.name == "T"
assert isinstance(first("type U<S extends string> = intrinsic;").typeAnnotation, S.TSIntrinsicKeyword)
interface = first("interface I<T> extends A.B<T>, C { a: T; }")
assert isinstance(interface.extends[0].expression, S.MemberExpression) and interface.extends[0].typeArguments
enum = first("declare const enum E { A, B = A + 1, 'C-d', [k] = 2 }")
assert enum.const and enum.declare and [m.computed for m in enum.body.members] == [False, False, False, True]
for text in ("import d, {a as b, type c} from 'm' with { type: 'json' };", "import type {T} from 'm';",
             "import * as ns from 'm';", "import 'side';", "import x = require('m');", "import y = A.B.C;",
             "export * as ns from 'm';", "export * from 'm';", "export {a as b, type c} from 'm';",
             "export type {T};", "export = x;", "export as namespace N;", "export default a + b;",
             "namespace A.B {\n    export const x = 1;\n}", "declare module \"short\";",
             "declare global {\n    interface W {}\n}", "type A<T> = T[];",
             "declare const enum E {\n    A,\n    B = A + 1,\n}",
             "export default (function() {});", "export default (a, b);"):
    same_text(text)
print("ok")

## TSPRS-07 · Types, with TypeScript's grouping where tree-sitter-typescript's differs

In [ ]:
def type_of(text):
    return first(f"type X = {text};").typeAnnotation


assert kinds(type_of("any | unknown | number | bigint | boolean | string | symbol | object | never | void | "
                     "undefined | null | this").types) == [
    "TSAnyKeyword", "TSUnknownKeyword", "TSNumberKeyword", "TSBigIntKeyword", "TSBooleanKeyword", "TSStringKeyword",
    "TSSymbolKeyword", "TSObjectKeyword", "TSNeverKeyword", "TSVoidKeyword", "TSUndefinedKeyword", "TSNullKeyword",
    "TSThisType"]
assert type_of("unique symbol").operator == "unique"
readonly = type_of("readonly A[] | B")  # `readonly` binds tighter than `|`
assert isinstance(readonly, S.TSUnionType) and readonly.types[0].operator == "readonly"
assert kinds(type_of("readonly A[] | B | C").types) == ["TSTypeOperator", "TSTypeReference", "TSTypeReference"]
assert isinstance(type_of("readonly A[] & B"), S.TSIntersectionType)
assert kinds(type_of("A | readonly B[] | C").types) == ["TSTypeReference", "TSTypeOperator", "TSTypeReference"]
assert type_of("keyof T").operator == "keyof"
assert [q.raw for q in type_of("`a${B}c`").quasis] == ["a", "c"]
assert isinstance(type_of("-1").literal, S.UnaryExpression) and type_of("'a'").literal.raw == "'a'"
tuple_type = type_of("[a: string, b?: number, ...c: boolean[]]").elementTypes
assert kinds(tuple_type) == ["TSNamedTupleMember", "TSNamedTupleMember", "TSRestType"] and tuple_type[1].optional
assert kinds(type_of("[string?, ...number[]]").elementTypes) == ["TSOptionalType", "TSRestType"]
assert isinstance(type_of("abstract new () => object"), S.TSConstructorType)
assert type_of("<T>(a: T) => a is string").returnType.typeAnnotation.parameterName.name == "a"
assert type_of("(a: unknown) => asserts a").returnType.typeAnnotation.asserts
mapped = type_of("{ -readonly [K in keyof T as `get${K & string}`]+?: T[K] }")
assert (mapped.readonly, mapped.optional) == ("-readonly", "+?") and mapped.nameType is not None
assert (type_of("{ readonly [K in T]-?: K }").readonly, type_of("{ [K in T]?: K }").optional) == ("readonly", "?")
assert type_of("{ +readonly [K in T]: K }").readonly == "+readonly"
conditional = type_of("T extends (infer U extends string)[] ? U : never")
assert conditional.extendsType.elementType.typeAnnotation.typeParameter.constraint is not None
assert type_of("typeof a.b<T>").typeArguments is not None
assert isinstance(type_of("typeof import('m')"), S.TSTypeQuery)
imported = type_of("import('m', { with: { 'resolution-mode': 'import' } }).A.B<T>")
assert isinstance(imported, S.TSImportType) and imported.options is not None and imported.typeArguments
assert TS.print(imported.qualifier) == "A.B"
assert isinstance(type_of("import('m')"), S.TSImportType) and type_of("import('m').X").qualifier.name == "X"
assert isinstance(first("let a: typeof import('m').x;").declarations[0].id.typeAnnotation.typeAnnotation.exprName,
                  S.TSImportType)
assert isinstance(type_of("A['b'][C]"), S.TSIndexedAccessType)
literal = type_of("{ a: string; readonly b?: number; c(): void; get d(): number; set d(v); (x): y; new (x): y; "
                  "[k: string]: unknown; static e: string }")
assert kinds(literal.members) == [
    "TSPropertySignature", "TSPropertySignature", "TSMethodSignature", "TSMethodSignature", "TSMethodSignature",
    "TSCallSignatureDeclaration", "TSConstructSignatureDeclaration", "TSIndexSignature", "TSPropertySignature"]
assert literal.members[1].readonly and literal.members[3].methodKind == "get"
assert isinstance(type_of("(A)"), S.TSParenthesizedType) and isinstance(type_of("A.B.C").typeName, S.TSQualifiedName)
assert isinstance(type_of("bigint"), S.TSBigIntKeyword)
assert isinstance(type_of("typeof this").exprName, S.ThisExpression)
assert type_of("new <T>() => T").typeParameters is not None
assert [m.typeAnnotation for m in type_of("{ a; b? }").members] == [None, None]
inferred = type_of("T extends (infer U)[] ? U : never").extendsType.elementType.typeAnnotation
assert inferred.typeParameter.constraint is None
assert isinstance(type_of("readonly string[]"), S.TSTypeOperator)
for text in ("type X = readonly A[] | B;", "type X = keyof T;", "type X = [a: string, b?: number, ...c: boolean[]];",
             "type X = abstract new () => object;",
             "type X = { -readonly [K in keyof T as `get${K & string}`]+?: T[K] };",
             "type X = T extends (infer U extends string)[] ? U : never;", "type X = typeof a.b<T>;",
             "type X = import('m').A.B<T>;", "type X = A['b'][C];", "type X = (A | B)[];", "type X = unique symbol;",
             "type X = `a${B}c`;", "type X = -1;", "type X = { a: string; readonly b?: number; c(): void; };",
             "type X = (() => void) | null;", "type X = <T>(a: T) => a is string;", "type X = typeof this;",
             "type X = new <T>() => T;", "type X = readonly string[];", "class X extends B implements a.b.C {}"):
    same_text(text)
print("ok")

## TSPRS-08 · JSX, with the `tsx` grammar: elements, fragments, attributes, children and the text between them

In [ ]:
element = value("<a.b x='1' y={2} {...z} c:d=\"e\" f>t &amp; u{v}{/* c */}{...w}<g /><>h</></a.b>", TSX)
assert isinstance(element.openingElement.name, S.JSXMemberExpression)
assert kinds(element.openingElement.attributes) == [
    "JSXAttribute", "JSXAttribute", "JSXSpreadAttribute", "JSXAttribute", "JSXAttribute"]
assert isinstance(element.openingElement.attributes[3].name, S.JSXNamespacedName)
assert element.openingElement.attributes[4].value is None
assert kinds(element.children) == ["JSXText", "JSXExpressionContainer", "JSXExpressionContainer", "JSXSpreadChild",
                                   "JSXElement", "JSXFragment"]
assert element.children[0].raw == "t &amp; u" and isinstance(element.children[2].expression, S.JSXEmptyExpression)
assert value("<C<T> />", TSX).openingElement.typeArguments is not None
assert value("<div>\n    a\n</div>", TSX).children[0].raw == "\n    a\n"
assert value("<div>  </div>", TSX).children[0].raw == "  " and value("<div></div>", TSX).children == []
assert value("<T,>(a: T) => a", TSX).typeParameters is not None
for text in ("<a.b x='1' y={2} {...z} f>t &amp; u{v}{...w}<g /><>h</></a.b>;", "<C<T> />;", "<div>{}</div>;",
             "const f = <T,>(a: T) => a;"):
    same_text(text, standard=TSX)
print("ok")

## TSPRS-09 · Comments are kept where statements and members are listed, on their own lines or trailing

In [ ]:
program = parse("""#!/usr/bin/env node
// one
a; /* two */
class C { // three
    // four
    m() {} // five
}
interface I { /* six */ a: string; }
enum E { A, // seven
}
switch (x) { case 1: // eight
}
f(/* dropped */ a);
""")
assert program.hashbang == "/usr/bin/env node"
assert kinds(program.body) == ["Comment", "ExpressionStatement", "Comment", "ClassDeclaration",
                               "TSInterfaceDeclaration", "TSEnumDeclaration", "SwitchStatement", "ExpressionStatement"]
assert [program.body[0].trailing, program.body[2].trailing, program.body[2].block] == [False, True, True]
assert [(c.text, c.trailing) for c in program.body[3].body.body if isinstance(c, S.Comment)] == [
    (" three", True), (" four", False), (" five", True)]
assert program.body[4].body.body[0].text == " six " and program.body[5].body.members[1].trailing
assert program.body[6].cases[0].consequent[0].trailing
same_text("""a; /* two */
class C { // three
    // four
    m() {} // five
}
enum E {
    A, // seven
}
type T = {
    // eight
    a: string;
};""")
print("ok")

## TSPRS-10 · What tree-sitter-typescript 0.23 cannot parse, which the pre-pass puts back

In [ ]:
klass = first("class C { accessor a = 1; static accessor b: number; accessor() {} accessor = 2; accessor }")
assert kinds(klass.body.body) == ["AccessorProperty", "AccessorProperty", "MethodDefinition", "PropertyDefinition",
                                  "PropertyDefinition"]
assert [m.key.name for m in klass.body.body] == ["a", "b", "accessor", "accessor", "accessor"]
assert klass.body.body[1].static
parameters = first("type T<in A, out B, in out C, const in D> = A;").typeParameters.params
assert [(p.isIn, p.isOut) for p in parameters] == [(True, False), (False, True), (True, True), (True, False)]
assert parameters[3].const and first("function f<in>() {}").typeParameters.params[0].name.name == "in"
assert first("import defer * as m from 'm';").phase == "defer" and first("import { defer } from 'm';").phase is None
assert first("import defer { a } from 'm';").phase == "defer"
assert first("export type * from 'm';").exportKind == "type"
assert first("export type * as ns from 'm';").exportKind == "type"
abstract = first("class C { abstract: number; readonly abstract?: string; abstract() {} }").body.body
assert [m.key.name for m in abstract] == ["abstract", "abstract", "abstract"]
assert value("using + 1").left.name == "using" and value("x = using").right.name == "using"
assert value("using in o").left.name == "using"
assert first("let x: import('m').A<B>;").declarations[0].id.typeAnnotation.typeAnnotation.typeArguments
spanning = parse("let x: import(\n'm').A<B>;\n// after\ny;")  # the import spans lines
assert spanning.body[0].declarations[0].id.typeAnnotation.typeAnnotation.typeArguments
assert not spanning.body[1].trailing  # lines are those written
called = value("a<import('m').B>()")
assert isinstance(called.typeArguments.params[0], S.TSImportType)
for text in ("class C {\n    accessor a = 1;\n    static accessor b: number;\n}", "type T<in A, out B, in out C> = A;",
             "import defer * as m from 'm';", "export type * from 'm';", "class C {\n    abstract: number;\n}",
             "using + 1;", "let x: import('m').A<B>;", "a<import('m').B>();"):
    same_text(text)
print("ok")

## TSPRS-11 · JavaScript: the editions of ECMAScript parse what has no TypeScript syntax, and reject the rest

In [ ]:
ES = ECMAScript2025.STANDARD
program = parse("const { a = 1 } = b;\nclass C { #x; static { } }\nlabel: for (const y of z) break label;", ES)
assert kinds(program.body) == ["VariableDeclaration", "ClassDeclaration", "LabeledStatement"]
assert isinstance(parse("<div />;", ECMAScript2025.JSX).body[0].expression, S.JSXElement)
fails("let a: number;", "TSTypeAnnotation is not ECMAScript, but this is ES2025", 1, 6, ES)
fails("enum E {}", "TSEnumDeclaration is not ECMAScript, but this is ES2025", 1, 1, ES)
fails("@d class C {}", "Decorator is not ECMAScript, but this is ES2025", 1, 1, ES)
fails("a ??= b;", "AssignmentExpression.operator '??=' needs ES2021, but this is ES2020", 1, 1,
      ECMAScript2020.STANDARD)
print("ok")

## TSPRS-12 · Errors name their line and column, counted in characters

In [ ]:
fails("let = ;", "syntax error", 1, 5)
fails("a\n😀; b +", "syntax error", 2, 4)  # columns count characters, not UTF-16 units
fails("if (a {}", "expected )", 1, 6)
fails("import defer * as m from 'm';", "ImportDeclaration.phase 'defer' needs TypeScript 5.9, but this is "
      "TypeScript 5.0", 1, 1, TypeScript50.STANDARD)
fails("<div />;", "syntax error", 1, 1)
fails("<T>x;", "syntax error", 1, 1, TSX)
fails("let s = 'é😀'; x satisfies T;", "TSSatisfiesExpression needs TypeScript 4.9, but this is TypeScript 4.8", 1,
      15,
      TypeScriptStandard(4, 8))
fails("type X = { [K in T]: K; a: string };", "unsupported syntax: index_signature", 1, 12)
fails("x = (a: T);", "unsupported syntax: parenthesized_expression", 1, 5)
fails("type X = ?string;", "unsupported syntax: flow_maybe_type", 1, 10)  # Flow's
fails("let x: import('m'", "syntax error", 1, 1)
print("ok")